# Machine Translation using Transformers

In [1]:


import os

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import TensorDataset

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.mps.is_available() else "cpu"
)

## English to Hindi

In [2]:
device

'mps'

In [3]:
dir_name = "indic_languages_corpus/bilingual/hi-en"

In [4]:
english = None
with open(os.path.join(os.path.abspath(dir_name), "train.en")) as f:
    english = f.readlines()

In [5]:
hindi = None
with open(os.path.join(os.path.abspath(dir_name), "train.hi")) as f:
    hindi = f.readlines()

In [108]:
X_train = english

In [117]:
y_train = hindi

In [110]:
english

['And what is their Sigil?\n',
 'I do not want to die.\n',
 "It's the same country I think.\n",
 "Then they'll be crying like babies.\n",
 '- No, I need power up!\n',
 'I will not eat him.\n',
 'You gotta get me to Charleston.\n',
 "- NO, HE'S NOT MY DAD.\n",
 'I told her we rest on Sundays.\n',
 "You could've at least informed me, right?\n",
 'Miporol, extremely potent, will keep you functioning normally until your death.\n',
 '- I am available.\n',
 'Ninety percent it is, Dr. Brand.\n',
 "Your little bitch says you're gonna put me in jail!\n",
 '- You can call me whatever you like.\n',
 "- You don't just kill a guy like that!\n",
 'You sent these?\n',
 'I really loved him.\n',
 "I ain't much at guessing games.\n",
 "Tell me you have Jor-El's memories, his conscience.\n",
 "You're sick and I can help you.\n",
 'Mike, do I get to ride with you?\n',
 'What do you fucking think?\n',
 'I know that woman you love also is ready to forgive you.\n',
 "Don't do it, man.\n",
 '- Say sorry right

##### Why not byte pair encoding or character by character or turn words into graph for autocomplete, or sentence translation or generation?

**We are designing infinte window transformers, handling millions or billions of sequences, we can easily do this**
###### Can we turn the data into graphs, since relationship between words is sparse same as wordnet

### Preprocessing

In [8]:
from collections import Counter

In [171]:
X_train = ['<sos> ' + line + ' <eos>' for line in english]


In [172]:
y_train = ['<sos> ' + line + ' <eos>' for line in hindi]

In [174]:
X_train[:3]



['<sos> And what is their Sigil?\n <eos>',
 '<sos> I do not want to die.\n <eos>',
 "<sos> It's the same country I think.\n <eos>"]

In [177]:
import unicodedata


# Converts the unicode file to ascii
# Since the model is dealing with multilingual text so it will be important to standardize the input text.
# Unicode normalization splits accented characters and replace compatibility characters with their ASCII equivalents.
# https://bit.ly/2TnLffX
def unicode_to_ascii(s):
    return "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )


def preprocess_sentence(w):
    w = unicode_to_ascii(w.lower().strip())

    # creating a space between a word and the punctuation following it
    # eg: "he is a boy." => "he is a boy ."

    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)

    # replacing everything with space except (a-z, A-Z, ".", "?", "!", ",")
    w = re.sub(r"[^a-zA-Z?.!,¿]+", " ", w)

    w = w.strip()

    # adding a start and an end token to the sentence
    # so that the model know when to start and stop predicting.
    w = "<sos> " + w + " <eos>"
    return w

In [180]:
X_train = [preprocess_sentence(w) for w in english]

In [175]:
import regex as re

corpus = [re.sub(r"[^a-zA-Z?.!,¿]+", " ", w.lower()) for t in X_train for w in t.rstrip().strip("-").split("\n")]

In [176]:
corpus[:3]

[' sos and what is their sigil?', ' eos ', ' sos i do not want to die.']

In [181]:
corpus = X_train

In [182]:
X_train[:5]

['<sos> and what is their sigil ? <eos>',
 '<sos> i do not want to die . <eos>',
 '<sos> it s the same country i think . <eos>',
 '<sos> then they ll be crying like babies . <eos>',
 '<sos> no , i need power up ! <eos>']

In [183]:
sub_corpus = X_train[:2]
corpus = X_train


# Converts the unicode file to ascii
# Since the model is dealing with multilingual text so it will be important to standardize the input text.
# Unicode normalization splits accented characters and replace compatibility characters with their ASCII equivalents.
def tokenize_and_embeddings(text):
    vocabulary = Counter()
    #stop_words= ['\n',]
    corpus = [a.strip().split() for a in text]

    for token in corpus:
        # Cannot do for hindi..
        vocabulary.update([t for t in token])
    word_to_index = {word: i + 1 for i, (word, _) in enumerate(vocabulary.items())}
    word_to_index["pad"] = 0
    numerical_sequences = [
        [word_to_index[token] for token in tokens] for tokens in corpus
    ]
    max_length = max(len(seq) for seq in numerical_sequences)

    padded_sequences = [
        seq + [word_to_index["pad"]] * (max_length - len(seq))
        for seq in numerical_sequences
    ]
    return padded_sequences, vocabulary, word_to_index

In [184]:
padded_sequences, vocabulary_train, train_index = tokenize_and_embeddings(X_train)

In [185]:
dict(sorted(train_index.items(), key=lambda x: x[1], reverse=True))

{'cancellation': 18986,
 'pouting': 18985,
 'retaining': 18984,
 'torino': 18983,
 'eau': 18982,
 'bouncing': 18981,
 'screenplay': 18980,
 'vacate': 18979,
 'naughtier': 18978,
 'method': 18977,
 'palatial': 18976,
 'vu': 18975,
 'deja': 18974,
 'requestingpermissiontoabort': 18973,
 'camaro': 18972,
 'manipulations': 18971,
 'subtlest': 18970,
 'rifleman': 18969,
 'malls': 18968,
 'whatdoyouwant': 18967,
 'thais': 18966,
 'tripped': 18965,
 'sjeea': 18964,
 'bumps': 18963,
 'exaggeration': 18962,
 'obelisk': 18961,
 'trollops': 18960,
 'wanton': 18959,
 'englishmen': 18958,
 'yaah': 18957,
 'adventurous': 18956,
 'prez': 18955,
 'wirelessly': 18954,
 'absolutes': 18953,
 'shovels': 18952,
 'triplearm': 18951,
 'moonbeam': 18950,
 'sinkhole': 18949,
 'odgodi': 18948,
 'nasumieno': 18947,
 'ney': 18946,
 'jour': 18945,
 'supportive': 18944,
 'catsup': 18943,
 'picturing': 18942,
 'volcano': 18941,
 'lawton': 18940,
 'bigwig': 18939,
 'mentals': 18938,
 'hayfield': 18937,
 'wantonly': 1

In [186]:
train_sequences, vocabulary, word_index_train = (
    padded_sequences,
    vocabulary_train,
    train_index,
)

In [187]:
target_sequences, vocabulary_target, target_index = tokenize_and_embeddings(y_train)

In [188]:
test_sequences, vocabulary_test, word_index_test = (
    target_sequences,
    vocabulary_target,
    target_index,
)

In [189]:
len(train_sequences)

84557

In [190]:
len(vocabulary_train)

18986

In [191]:
len(vocabulary_test)

40398

In [192]:
vocabulary_train["pad"] = 0
vocabulary_target["pad"] = 0

In [193]:
vocabulary = vocabulary_train
vocabulary_test = vocabulary_target

In [194]:
len(word_index_test)

40399

In [195]:
len(word_index_train), len(vocabulary_train)

(18986, 18986)

In [196]:
vocabulary_train.most_common(40)

[('<sos>', 84557),
 ('<eos>', 84557),
 ('.', 74449),
 (',', 29349),
 ('you', 21860),
 ('i', 18475),
 ('the', 16872),
 ('?', 14523),
 ('to', 12322),
 ('s', 10636),
 ('a', 10364),
 ('it', 9379),
 ('!', 8025),
 ('and', 6794),
 ('that', 6785),
 ('t', 6545),
 ('of', 6072),
 ('we', 5954),
 ('is', 5512),
 ('in', 5398),
 ('me', 4946),
 ('this', 4552),
 ('he', 4198),
 ('what', 4169),
 ('my', 3981),
 ('your', 3902),
 ('for', 3883),
 ('have', 3547),
 ('re', 3435),
 ('on', 3433),
 ('not', 3283),
 ('do', 3238),
 ('be', 3163),
 ('m', 3073),
 ('are', 3032),
 ('can', 2969),
 ('don', 2784),
 ('no', 2745),
 ('was', 2735),
 ('all', 2628)]

In [197]:
vocabulary.most_common(10)

[('<sos>', 84557),
 ('<eos>', 84557),
 ('.', 74449),
 (',', 29349),
 ('you', 21860),
 ('i', 18475),
 ('the', 16872),
 ('?', 14523),
 ('to', 12322),
 ('s', 10636)]

In [198]:
torch.mps.is_available()

True

In [24]:
device

'mps'

In [25]:
torch.__version__

'2.13.0'

In [26]:
torch.mps.is_available()

True

In [27]:
X_train[:2]

['And what is their Sigil?\n', 'I do not want to die.\n']

In [28]:
y_train[:2]

['और उनके Sigil क्या है?\n', 'मैं मरना नहीं चाहता.\n']

In [28]:
#train_sequences[:2]

In [199]:
len(vocabulary_train), len(vocabulary_test), len(word_index_train), len(word_index_test)

(18986, 40399, 18986, 40399)

In [200]:
#todo positional encoding relative and absolute

In [201]:
VOCAB_SIZE = len(vocabulary)
BATCH_SIZE = 64

embedding_dim = 256
# no of GRUs
units = 1024
vocab_inp_size = len(train_index)
vocabulary_target_size = len(target_index)

In [202]:
VOCAB_SIZE

18986

In [225]:
X_train = torch.tensor(train_sequences, dtype=torch.long)
y_train = torch.tensor(target_sequences, dtype=torch.long)

In [34]:
#X_train[:2]

In [30]:
# import pandas as pd
# df = pd.DataFrame.from_dict({'sentences':corpus,'numerical':numerical_sequences})

In [31]:
# df

## Model Building

In [204]:
class AttentionHead(nn.Module):
    """
    Multi Attention Head
    Splits Input in Q,K,V

    """

    def __init__(self, model_dim, H, dropout_rate=0.1):
        super().__init__()
        self.Wq = nn.Linear(model_dim, model_dim)
        self.Wk = nn.Linear(model_dim, model_dim)
        self.Wv = nn.Linear(model_dim, model_dim)
        self.H = H
        self.d_h = int(model_dim / H)
        self.dropout = nn.Dropout(p=dropout_rate)

        self.Wo = nn.Linear(model_dim, model_dim)

    def forward(self, sequences, attn_mask=False):
        """Input shape: [batch_size, seq_len, d_model=num_head * d_head]
        # if key_value_states are provided this layer is used as a cross-attention layer for text translation..

        # for the decoder

        """
        batch_size, seq_len, model_dim = sequences.size()
        Q = self.Wq(sequences)

        K, V = self.Wk(sequences), self.Wv(sequences)

        A = Q @ K.transpose(-2, -1)
        if attn_mask is not None and attn_mask:
            A = A.masked_fill(attn_mask == 0, -float("inf"))
        A = F.softmax(A / self.d_h ** 0.5, dim=-1, )  # Applying softmax
        #todo A_causal = F.scaled_dot_product_attention(Q,K,V,is_causal=True)
        A = self.dropout(A)  # Final

        # Output Z

        Z = A @ V  # torch,tensor
        print(f"{Z.shape=}")
        ### Concatenating in parallel along heads and sequences
        ## Because continuous input
        # Z = (Z.contiguous().view(
        #     batch_size,seq_len,self.H*self.d_h)
        # )
        # 2. Transpose to move seq_len before H: shape (batch_size, seq_len, H, d_h)
        # NOTE: Z is now NON-CONTIGUOUS in memory!
        Z = Z.transpose(1, 2).reshape(batch_size, seq_len, model_dim)
        # 3. Concatenate all heads into d_model = H * d_h
        # Fails without .contiguous():
        # Z = Z.contiguous().view(batch_size,seq_len,self.H*self.d_h)
        # final linear projections
        Z = self.Wo(Z)
        return A, Z

In [205]:
vocabulary["pad"] = 0

### Debug

In [206]:
batch_size = 64
sequence_length = 200
model_dim = 2048
d_model = 512
d_ff = 2048
H = 8
dropout = 0.1

In [207]:
model = AttentionHead(d_model, H)

In [208]:
len(padded_sequences[0])
dim = len(padded_sequences[0])

In [209]:
model

AttentionHead(
  (Wq): Linear(in_features=512, out_features=512, bias=True)
  (Wk): Linear(in_features=512, out_features=512, bias=True)
  (Wv): Linear(in_features=512, out_features=512, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (Wo): Linear(in_features=512, out_features=512, bias=True)
)

In [210]:
x = torch.randn(batch_size, sequence_length, d_model)

In [211]:
Z = model(x)

Z.shape=torch.Size([64, 200, 512])


In [212]:
torch.save(model, 'attention_head.pt')

In [44]:
# len(long_tokenized)

In [45]:
len(vocabulary)

49763

In [42]:
# long_tokenized[:2].shape

In [43]:
# long_tokenized.shape

In [45]:
# len(vocabulary.values())

## Feed Forward and Attention Residual

In [213]:
batch, sentence_length, embedding_dim = 20, 10, 8
embedding = torch.randn(batch, sentence_length, embedding_dim)
layer_norm = nn.LayerNorm(embedding_dim)

In [214]:
x = layer_norm(embedding)

In [215]:
x

tensor([[[-1.3407, -1.5695,  0.7254,  ..., -0.7226,  0.1066,  1.0561],
         [ 0.2612,  1.3169, -1.7222,  ..., -0.1342, -0.8799,  1.2471],
         [ 0.3306, -2.0598, -0.4181,  ...,  1.1099,  0.2077,  0.3591],
         ...,
         [ 0.6767,  1.6068,  0.7828,  ..., -1.5386, -0.7660,  0.1403],
         [ 0.0128,  0.8613,  0.1277,  ..., -1.8595,  1.3160,  0.7642],
         [ 0.6214, -1.5053, -0.3224,  ...,  1.2396, -1.6754,  0.5522]],

        [[ 0.3619,  0.2161, -1.3174,  ..., -0.2338, -1.1438,  1.9921],
         [ 1.9571,  0.9380, -0.6003,  ..., -0.8505, -0.9761, -0.7954],
         [ 0.2478, -0.5256,  1.2528,  ..., -0.9122,  0.4292,  1.6522],
         ...,
         [ 0.7714,  0.5515,  0.9909,  ..., -1.2302, -1.0263,  0.8335],
         [-1.8221, -0.2518, -0.8638,  ...,  1.7446,  0.4704,  0.5581],
         [-0.8935, -0.9175, -1.6704,  ...,  0.9344,  0.9199,  0.1884]],

        [[ 1.0550,  0.2833, -2.2047,  ..., -0.6972, -0.3192,  0.3853],
         [ 1.2938,  0.7180,  0.3665,  ..., -0

In [224]:
embedding = nn.Embedding(num_embeddings=len(vocabulary_train), embedding_dim=256, padding_idx=0)

In [228]:
embedding(X_train[len(vocabulary_train)])

tensor([[-0.1650, -1.6186, -1.3936,  ...,  0.2282,  1.2731,  1.3358],
        [-0.7178, -1.3265,  0.5525,  ..., -0.6155, -0.8562,  1.0619],
        [ 1.6397,  1.1433, -1.2694,  ...,  0.2391, -1.7891, -1.2526],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
       grad_fn=<EmbeddingBackward0>)

In [217]:
embedding_sample = nn.Embedding(num_embeddings=10, embedding_dim=3, padding_idx=0)

In [218]:
input = torch.LongTensor([[0, 2, 0, 5]])

In [ ]:
layer_norm(train_sequences[0]).shape

In [221]:
X_train_model = torch.tensor(train_sequences, dtype=torch.long)

In [222]:
X_train_model

tensor([[  1,   2,   3,  ...,   0,   0,   0],
        [  1,   9,  10,  ...,   0,   0,   0],
        [  1,  16,  17,  ...,   0,   0,   0],
        ...,
        [  1,   9, 202,  ...,   0,   0,   0],
        [  1, 760,  30,  ...,   0,   0,   0],
        [  1,  38,  84,  ...,   0,   0,   0]])

In [ ]:
embedding_sample = embedding_sample(X_train_model)

In [59]:
embedding_sample

tensor([[[ 0.0000,  0.0000,  0.0000],
         [ 0.2011,  1.6136, -0.2915],
         [ 0.0000,  0.0000,  0.0000],
         [ 1.3478,  0.1949,  1.5027]]], grad_fn=<EmbeddingBackward0>)

In [60]:
class FFN(nn.Sequential):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.layer1 = nn.Linear(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_ff)
        self.activation = nn.GELU()
        self.out = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.layer1(x)
        x = self.norm1(x)
        x = self.activation(x)
        x = self.out(x)
        return x

In [61]:
ffn = FFN(d_model, d_ff)

In [62]:
ffn

FFN(
  (layer1): Linear(in_features=512, out_features=2048, bias=True)
  (norm1): LayerNorm((2048,), eps=1e-05, elementwise_affine=True, bias=True)
  (activation): GELU(approximate='none')
  (out): Linear(in_features=2048, out_features=512, bias=True)
)

In [63]:
ffn = nn.Sequential(
    nn.Linear(d_model, d_ff), nn.LayerNorm(d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
)

In [64]:
x.shape, d_ff, d_model

(torch.Size([20, 10, 8]), 2048, 512)

In [65]:
class Transformer(nn.Module):
    def __init__(self, d_model, d_ff, num_head, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_ff = d_ff
        self.H = num_head
        self.dropout = nn.Dropout(p=dropout)

        self.attention = AttentionHead(d_model, num_head, dropout)
        self.ffn = ffn(d_model, d_ff)

    def forward(self, x, attn_mask=False):
        Z = self.attention(x, attn_mask)
        Z = self.dropout(Z)
        Z = Z + x
        Z = self.ffn(Z) + x
        return Z

In [66]:
corpus = X_train

In [67]:
embedding = nn.Embedding(10, 3, padding_idx=0)
input = torch.LongTensor([[0, 2, 0, 5]])
embedding(input).shape

torch.Size([1, 4, 3])

In [68]:
embedding(input)

tensor([[[ 0.0000,  0.0000,  0.0000],
         [-0.3457,  2.5271, -0.3757],
         [ 0.0000,  0.0000,  0.0000],
         [ 1.0943, -0.9798,  0.3185]]], grad_fn=<EmbeddingBackward0>)

## Encoder Decoder

In [69]:
d_in = len(corpus)

In [64]:
# encoder = nn.Linear(long_tokenized.shape[0],long_tokenized.shape[1])

In [70]:
embedding

Embedding(10, 3, padding_idx=0)

### Constants

In [229]:
VOCAB_SIZE = len(vocabulary)
BATCH_SIZE = 64

embedding_dim = 256
# no of GRUs
units = 1024
EPOCHS = 30
vocab_inp_size = len(vocabulary)
vocabulary_target_size = len(vocabulary_target)

In [230]:
len(train_index)

18986

### Data

In [231]:
from torch.utils.data import Dataset, DataLoader

In [232]:
dataset = TensorDataset(X_train, y_train)

In [233]:
X_train[0]

tensor([1, 2, 3, 4, 5, 6, 7, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [234]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 6, 7, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 tensor([1, 2, 3, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0]))

In [77]:
# todo convert to graph
class Hindi_English(Dataset):
    def __init__(self):
        pass

## Model Building

In [235]:
data_loader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=2)

In [236]:
x, y = next(iter(data_loader))

In [237]:
len(data_loader)

1322

In [238]:
x

tensor([[  1,   2,   3,  ...,   0,   0,   0],
        [  1,   9,  10,  ...,   0,   0,   0],
        [  1,  16,  17,  ...,   0,   0,   0],
        ...,
        [  1, 221,  30,  ...,   0,   0,   0],
        [  1, 185,  18,  ...,   0,   0,   0],
        [  1,  61, 225,  ...,   0,   0,   0]])

### Gru Operation

In [239]:
n, d, m = 3, 5, 7
embedding = nn.Embedding(n, d, max_norm=1.0)
print(f"{embedding.weight.shape=}")
W = torch.randn((m, d), requires_grad=True)
print(f"{W.shape=}")
idx = torch.tensor([1, 2])
a = (
        embedding.weight.clone() @ W.t()
)  # weight must be cloned for this to be differentiable
print(f"{a.shape=},{(a.unsqueeze(0)).shape=}")
b = embedding(idx) @ W.t()  # modifies weight in-place
print(f"{b.shape},{(b.unsqueeze(1)).shape=}")
out = a.unsqueeze(0) + b.unsqueeze(1)
loss = out.sigmoid().prod()
print(loss.backward())

embedding.weight.shape=torch.Size([3, 5])
W.shape=torch.Size([7, 5])
a.shape=torch.Size([3, 7]),(a.unsqueeze(0)).shape=torch.Size([1, 3, 7])
torch.Size([2, 7]),(b.unsqueeze(1)).shape=torch.Size([2, 1, 7])
None


In [241]:
loss.item()

8.272425354295125e-41

In [242]:
out.shape

torch.Size([2, 3, 7])

In [243]:
rnn = nn.GRU(input_size=10, hidden_size=20, num_layers=2)
input = torch.randn(2, 3, 10)
#embedding_inp = nn.Embedding()
h0 = torch.randn(2, 3, 20)  # first if batch size or number of layers
output, hn = rnn(input, h0)
print(f"{output.shape=},{h0.shape=}")

output.shape=torch.Size([2, 3, 20]),h0.shape=torch.Size([2, 3, 20])


### Encoder

In [293]:



class Encoder(nn.Module):
    def __init__(self, batch_sz, embedding_dim, enc_units, vocab_size):
        super().__init__()
        self.batch_sz = batch_sz  # set batch size
        self.enc_units = enc_units  # set the number of GRU units
        self.embedding_layer = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        # self.hidden_state=256
        self.gru = nn.GRU(embedding_dim, self.enc_units, batch_first=True)

    def forward(self, x, hidden=None):
        print(f"{x.shape}")
        x = self.embedding_layer(x)
        print(f"Embedded: {x.shape=}")
        # assert h==sample_hidden
        output, state = self.gru(x, hidden)
        return output, state

    def initialize_hidden(self):
        return torch.zeros(1, self.batch_sz, self.enc_units)

### Decoder

In [294]:
class Decoder(nn.Module):
    def __init__(self, batch_sz, embedding_dim, dec_units, vocab_size):
        super().__init__()
        self.batch_sz = batch_sz  # batch_size which is defined as 64
        self.dec_units = dec_units  # the number of decoder GRU units
        self.embedding_layer = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0

        )
        self.gru = nn.GRU(embedding_dim, self.dec_units)
        self.out = nn.Linear(dec_units, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding_layer(x)
        output, state = self.gru(x, hidden)
        print(f"{output.shape=},{state.shape=}")

        logits = self.out(output)
        return logits, state

### Debug


In [246]:
encoder = Encoder(BATCH_SIZE, embedding_dim, units, vocab_inp_size)

In [278]:
vocab_inp_size

18986

In [247]:
encoder

Encoder(
  (embedding_layer): Embedding(18986, 256)
  (gru): GRU(256, 1024, batch_first=True)
)

In [248]:
sample_hidden = encoder.initialize_hidden()

In [249]:
print(f"f{sample_hidden.shape=}")

fsample_hidden.shape=torch.Size([1, 64, 1024])


In [250]:
65536 / 64 / 16

64.0

In [251]:
out, state = encoder(x)
print(f"{out.shape=},{state.shape=}")

torch.Size([64, 72])
Embedded: x.shape=torch.Size([64, 72, 256])
out.shape=torch.Size([64, 72, 1024]),state.shape=torch.Size([1, 64, 1024])


In [252]:
x.shape

torch.Size([64, 72])

In [253]:
out1, state1 = encoder(x, sample_hidden)
print(f"{out1.shape=},{state1.shape=}")

torch.Size([64, 72])
Embedded: x.shape=torch.Size([64, 72, 256])
out1.shape=torch.Size([64, 72, 1024]),state1.shape=torch.Size([1, 64, 1024])


In [254]:
decoder = Decoder(BATCH_SIZE, 1024, units, len(vocabulary_target))

In [255]:
sample_hidden = encoder.initialize_hidden()

In [256]:
sample_hidden.shape

torch.Size([1, 64, 1024])

In [257]:
x.shape

torch.Size([64, 72])

In [258]:
import numpy as np

uniform = np.random.uniform(1, (BATCH_SIZE, 1))
print(f"{uniform.shape=}")
x1 = torch.rand(BATCH_SIZE, 2).uniform_(-3, 3)
x1

uniform.shape=(2,)


tensor([[ 0.5834,  1.0526],
        [ 1.4969, -0.4367],
        [-2.0706,  2.0456],
        [-1.9103,  0.2828],
        [-1.7039,  2.8109],
        [-2.3030,  1.8296],
        [-0.9041,  1.1352],
        [ 2.0050,  1.3634],
        [-1.0072, -2.9693],
        [-0.9140, -1.9471],
        [ 2.6781, -2.5638],
        [ 0.4580,  0.4733],
        [-0.9406, -1.0056],
        [-2.6821, -1.6548],
        [-1.5400,  1.9869],
        [ 0.1794,  0.0168],
        [ 1.2145,  0.8454],
        [-2.7234, -1.3920],
        [ 1.1040, -1.1967],
        [ 2.7558,  1.3152],
        [ 1.1745,  2.1549],
        [-1.3152,  1.3043],
        [-2.5979,  1.8458],
        [ 2.0218, -0.6212],
        [-0.9579,  2.1663],
        [-2.6597, -2.1337],
        [ 0.1255, -1.2319],
        [-2.8840,  1.7061],
        [ 2.4921, -2.2508],
        [-2.2023, -2.8993],
        [ 0.8301, -0.4144],
        [ 1.2395, -0.8005],
        [ 2.3832, -0.4758],
        [-0.6509,  1.5948],
        [-2.1966, -0.7827],
        [-2.8684, -1

In [259]:
uniform.ndim

1

In [129]:
encoder

Encoder(
  (embedding_layer): Embedding(49763, 256)
  (gru): GRU(256, 1024, batch_first=True)
)

In [130]:
list(encoder.named_modules())

[('',
  Encoder(
    (embedding_layer): Embedding(49763, 256)
    (gru): GRU(256, 1024, batch_first=True)
  )),
 ('embedding_layer', Embedding(49763, 256)),
 ('gru', GRU(256, 1024, batch_first=True))]

In [131]:
x1.ndim

2

In [132]:
x1 = torch.tensor(x1.detach().clone(), dtype=torch.long)

/var/folders/px/m0g9wbyn1sv678fsx9lgsm600000gn/T/ipykernel_25003/3818009117.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x1 = torch.tensor(x1.detach().clone(), dtype=torch.long)


In [133]:
x1.ndim

2

In [134]:
sample_decoder_output, state2 = decoder(torch.tensor(uniform, dtype=torch.long))
print(sample_decoder_output.shape, state2.shape)

output.shape=torch.Size([2, 1024]),state.shape=torch.Size([1, 1024])
torch.Size([2, 40397]) torch.Size([1, 1024])


In [135]:
decoder

Decoder(
  (embedding_layer): Embedding(40397, 1024)
  (gru): GRU(1024, 1024)
  (out): Linear(in_features=1024, out_features=40397, bias=True)
)

In [137]:
list(decoder.state_dict().keys())

['embedding_layer.weight',
 'gru.weight_ih_l0',
 'gru.weight_hh_l0',
 'gru.bias_ih_l0',
 'gru.bias_hh_l0',
 'out.weight',
 'out.bias']

In [139]:
sample_decoder_output2, state2 = decoder(
    torch.tensor(np.random.uniform(3, (BATCH_SIZE, 3)), dtype=torch.long),
)
sample_decoder_output2

output.shape=torch.Size([2, 1024]),state.shape=torch.Size([1, 1024])


tensor([[-0.0764, -0.2132, -0.1000,  ...,  0.0529,  0.1691, -0.0989],
        [-0.1789, -0.1439, -0.1014,  ...,  0.0240,  0.1971, -0.1708]],
       grad_fn=<AddmmBackward0>)

In [140]:
print(sample_decoder_output2.shape, state2.shape)

torch.Size([2, 40397]) torch.Size([1, 1024])


In [141]:
sample_decoder_output1, state1 = decoder(torch.tensor(uniform, dtype=torch.long))

output.shape=torch.Size([2, 1024]),state.shape=torch.Size([1, 1024])


In [142]:
len(data_loader)

1322

In [143]:
x

tensor([[  1,   2,   3,  ...,   0,   0,   0],
        [  6,   7,   8,  ...,   0,   0,   0],
        [ 12,  13,  14,  ...,   0,   0,   0],
        ...,
        [256,  61, 257,  ...,   0,   0,   0],
        [259,  13, 260,  ...,   0,   0,   0],
        [261, 262,   0,  ...,   0,   0,   0]])

In [144]:
y

tensor([[  1,   2,   3,  ...,   0,   0,   0],
        [  6,   7,   8,  ...,   0,   0,   0],
        [ 10,  11,  12,  ...,   0,   0,   0],
        ...,
        [248, 166, 180,  ...,   0,   0,   0],
        [250,  36, 251,  ...,   0,   0,   0],
        [252, 253,   0,  ...,   0,   0,   0]])

In [145]:
sample_hidden = encoder.initialize_hidden()

In [146]:
sample_hidden.shape

torch.Size([1, 64, 1024])

In [155]:
sample_hidden = sample_hidden.view(1, sample_hidden.shape[-1], sample_hidden.shape[-2])

In [156]:
sample_hidden.shape

torch.Size([1, 1024, 64])

In [157]:
x.shape

torch.Size([64, 56])

In [158]:
x_seq = torch.tensor([[1.0] * 5, [2.0] * 5, [3.0] * 5])
x_seq.shape

torch.Size([3, 5])

In [159]:
x_seq.reshape(1, 3, 5)

tensor([[[1., 1., 1., 1., 1.],
         [2., 2., 2., 2., 2.],
         [3., 3., 3., 3., 3.]]])

In [160]:
x.shape

torch.Size([64, 56])

In [161]:
batch_size

64

In [162]:
x_batched = x.view(batch_size, 1, x.shape[-1])

In [163]:
x_batched.shape

torch.Size([64, 1, 56])

In [164]:
X_train.shape

torch.Size([84557, 56])

In [282]:
encoder = Encoder(BATCH_SIZE, embedding_dim, units, vocab_inp_size)

In [ ]:
# sample_output, sample_hidden = encoder(x, sample_hidden)

In [260]:


from dataclasses import dataclass


@dataclass(init=True, repr=True)
class InventoryItem:
    """Class for keeping track of an item in inventory."""

    name: str
    unit_price: float
    quantity_on_hand: int = 0

    def total_cost(self) -> float:
        return self.unit_price * self.quantity_on_hand


item = InventoryItem(name="T shirt", unit_price=10, quantity_on_hand=3)
print(item, item.total_cost())

InventoryItem(name='T shirt', unit_price=10, quantity_on_hand=3) 30


In [261]:

input = torch.randn(3, 2, requires_grad=True)
target = torch.rand(3, 2, requires_grad=False)
loss = F.binary_cross_entropy(torch.sigmoid(input), target)
loss.backward()
print(loss.item())

0.8955626487731934


In [262]:
batch_size

64

In [263]:
encoder

Encoder(
  (embedding_layer): Embedding(18986, 256)
  (gru): GRU(256, 1024, batch_first=True)
)

## Training

In [264]:
from typing import List
from dataclasses import dataclass


@dataclass(init=True)
class Result:
    train_loss: List[float]
    test_loss: List[float]

    train_acc: List[float]
    test_acc: List[float]

In [265]:
dir(encoder)

['T_destination',
 '__annotate_func__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_apply',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_buffers',
 '_call_impl',
 '_compiled_call_impl',
 '_forward_hooks',
 '_forward_hooks_always_called',
 '_forward_hooks_with_kwargs',
 '_forward_pre_hooks',
 '_forward_pre_hooks_with_kwargs',
 '_get_backward_hooks',
 '_get_backward_pre_hooks',
 '_get_name',
 '_is_full_backward_hook',
 '_load_from_state_dict',
 '_load_state_dict_post_hooks',
 '_load_state_dict_pre_hooks',
 '_maybe_warn_non_full_backward_hook',
 '_modules',

In [266]:
encoder = encoder.to(device)

In [267]:
decoder = decoder.to(device)

In [306]:
loss = torch.nn.BCEWithLogitsLoss()
loss_object = torch.nn.BCEWithLogitsLoss()
criterion = torch.optim.Adam(encoder.parameters(), lr=0.3)
lr_scheduler = torch.optim.lr_scheduler  #todo

In [299]:
X_train[0].shape

torch.Size([72])

In [269]:
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

In [270]:
for p, m in encoder.named_parameters():
    print(f"{p},{m.shape=}")

embedding_layer.weight,m.shape=torch.Size([18986, 256])
gru.weight_ih_l0,m.shape=torch.Size([3072, 256])
gru.weight_hh_l0,m.shape=torch.Size([3072, 1024])
gru.bias_ih_l0,m.shape=torch.Size([3072])
gru.bias_hh_l0,m.shape=torch.Size([3072])


In [174]:
len(loader)

1321

In [295]:
@torch.compile(options={"triton.cudagraphs": True}, fullgraph=True)
def foo(x):
    return torch.sin(x) + torch.cos(x)

In [296]:
foo(torch.tensor([2]))

tensor([0.4932])

In [283]:
x, y = next(iter(loader))

In [285]:
x.shape

torch.Size([64, 72])

In [303]:
a = torch.logical_not(torch.tensor([1, 2, 3]))

In [305]:
def loss_function(real, pred):
    mask = torch.logical_not(torch.eq(real, 0))
    loss_ = loss_object(real, pred)
    mask = mask.to(dtype=loss_.dtype)
    loss_ *= mask
    return torch.sum(mask) / (torch.sum(loss_) + 1e-5)


In [240]:
from tqdm.auto import tqdm


@torch.compile
def train_epoch(epoch: int, model: nn.Module, train_loader: torch.utils.data.DataLoader, optim: torch.optim.Optimizer):
    model.train()


criterion = nn.CrossEntropyLoss()
for batched_input, batched_target in tqdm(
        train_loader, desc=f"Training @ epoch {epoch}"
):
    batched_input, batched_target = batched_input.to(device), batched_target.to(device)
optim.zero_grad()
loss = criterion(model(batched_input), batched_targ,





In [308]:
word_index_test['<sos>']

1

In [317]:
list((w, p.shape) for w, p in encoder.named_parameters() if p.requires_grad)

[('embedding_layer.weight', torch.Size([18986, 256])),
 ('gru.weight_ih_l0', torch.Size([3072, 256])),
 ('gru.weight_hh_l0', torch.Size([3072, 1024])),
 ('gru.bias_ih_l0', torch.Size([3072])),
 ('gru.bias_hh_l0', torch.Size([3072]))]

In [ ]:
@torch.compile
def train(inp, target, enc_hidden, word_index_test):
    loss = 0
    encoder_output, enc_hidden = encoder(inp, enc_hidden)
    dec_hidden = enc_hidden
    dec_output = torch.unsqueeze(word_index_test)

### Sample Embedding

In [292]:
x.shape

torch.Size([64, 72])

In [286]:
encoder_output, encoder_hidden = encoder(x)

torch.Size([64, 72])
Embedded: x.shape=torch.Size([64, 72, 256])


In [288]:
sample_hidden = encoder.initialize_hidden()

In [290]:
encoder_output.shape

torch.Size([64, 72, 1024])

In [291]:
encoder_hidden.shape

torch.Size([1, 64, 1024])

In [ ]:
loss_object

In [301]:
class

Training:   0%|          | 0/1321 [00:00<?, ?it/s]

In [ ]:
def loop(model, epoch, )